# NAIP ZIP Organizer: Prefire/Postfire + Unzip

This notebook organizes NAIP imagery ZIP files into `prefire` and `postfire` folders based on the acquisition date encoded in each ZIP filename, then unzips each file into a folder named after the ZIP filename stem.

## What this does

1. Takes a list of folder paths that contain NAIP ZIP files.
2. Parses acquisition date from filenames like `m_3611835_se_11_060_20220707.ZIP` (last underscore token = `YYYYMMDD`).
3. Classifies files as `prefire` or `postfire` from a fire year.
4. Moves ZIP files into `prefire` / `postfire` subfolders.
5. Unzips each ZIP into a same-name folder in its destination folder.

By default, files with acquisition year `< fire_year` are `prefire`; files with acquisition year `>= fire_year` are `postfire`.

In [6]:
from pathlib import Path
import shutil
import zipfile
from datetime import datetime, date

# Folders that contain NAIP ZIP files (edit this list).
NAIP_FOLDERS = [
    # Path('./downloads/castle/naip'),
    Path('./downloads/creek/naip'),
    Path('./downloads/czu/naip'),
    Path('./downloads/northcomplex/naip'),
]

# Fire year mapping (keyed by folder path string).
# You can define per-folder fire years or use DEFAULT_FIRE_YEAR.
FIRE_YEAR_BY_FOLDER = {
    './downloads/castle/naip': 2020,
    # 'notebooks/downloads/creek/naip': 2020,
    # 'notebooks/downloads/czu/naip': 2020,
    # 'notebooks/downloads/northcomplex/naip': 2020,
}
DEFAULT_FIRE_YEAR = 2020  # Example: 2020

# Optional per-folder exact fire date (YYYY-MM-DD).
# If provided, this takes precedence over year-based classification.
FIRE_DATE_BY_FOLDER = {
    # './downloads/castle/naip': '2020-08-16',
}

# Year-based rule fallback when exact fire date is not provided.
# True means imagery from the fire year is treated as prefire.
SAME_YEAR_IS_PREFIRE = True

# Safety limits for sanity checks before move/unzip.
MAX_MOVE_FILES = 5000
MAX_UNZIP_FILES = 5000
MAX_ESTIMATED_UNZIP_GB = 200.0

# If True, print planned operations but do not move or unzip.
DRY_RUN = False

In [7]:
def parse_acquisition_date_from_name(zip_path: Path) -> datetime:
    """Parse acquisition date from final underscore token (YYYYMMDD)."""
    stem = zip_path.stem
    date_token = stem.rsplit('_', 1)[-1]
    return datetime.strptime(date_token, '%Y%m%d')


def resolve_fire_year(folder: Path) -> int:
    folder_key = str(folder)
    fire_year = FIRE_YEAR_BY_FOLDER.get(folder_key, DEFAULT_FIRE_YEAR)
    if fire_year is None:
        raise ValueError(
            f'No fire year configured for {folder_key}. Add it to FIRE_YEAR_BY_FOLDER or set DEFAULT_FIRE_YEAR.'
        )
    return int(fire_year)


def resolve_fire_date(folder: Path):
    folder_key = str(folder)
    fire_date_text = FIRE_DATE_BY_FOLDER.get(folder_key)
    if not fire_date_text:
        return None
    return datetime.strptime(fire_date_text, '%Y-%m-%d').date()


def classify_prefire_postfire(acq_date: datetime, fire_year: int, fire_date: date | None = None) -> str:
    acq_day = acq_date.date()
    if fire_date is not None:
        # Acquisition on or before fire date is treated as prefire.
        return 'prefire' if acq_day <= fire_date else 'postfire'

    if SAME_YEAR_IS_PREFIRE:
        return 'prefire' if acq_date.year <= fire_year else 'postfire'
    return 'prefire' if acq_date.year < fire_year else 'postfire'


def find_zip_files(folder: Path):
    # Recursive search for .zip/.ZIP while skipping already-organized output folders.
    for candidate in folder.rglob('*'):
        if not candidate.is_file():
            continue
        if candidate.suffix.lower() != '.zip':
            continue
        if {'prefire', 'postfire'}.intersection(set(candidate.parts)):
            continue
        yield candidate


def find_bucket_zip_files(folder: Path, bucket: str):
    bucket_dir = folder / bucket
    if not bucket_dir.exists():
        return
    for candidate in bucket_dir.glob('*'):
        if candidate.is_file() and candidate.suffix.lower() == '.zip':
            yield candidate


def gather_zip_candidates_for_classification(folder: Path):
    # Include unorganized ZIPs and already bucketed ZIPs so reruns can fix misclassification.
    seen = set()
    for candidate in find_zip_files(folder):
        seen.add(candidate)
    for bucket in ['prefire', 'postfire']:
        for candidate in find_bucket_zip_files(folder, bucket):
            seen.add(candidate)
    return sorted(seen)


def human_size(num_bytes: int) -> str:
    units = ['B', 'KB', 'MB', 'GB', 'TB']
    size = float(num_bytes)
    for unit in units:
        if size < 1024 or unit == units[-1]:
            return f'{size:.2f} {unit}'
        size /= 1024


def zip_storage_stats(zip_path: Path) -> tuple[int, int]:
    """Return (compressed_bytes, uncompressed_bytes) from ZIP metadata."""
    compressed = zip_path.stat().st_size
    uncompressed = 0
    with zipfile.ZipFile(zip_path, 'r') as zf:
        for info in zf.infolist():
            uncompressed += info.file_size
    return compressed, uncompressed

In [8]:
move_plan = []
planning_skips = 0
already_in_correct_bucket = 0
total_move_zip_bytes = 0

for base_folder in NAIP_FOLDERS:
    base_folder = Path(base_folder)
    if not base_folder.exists():
        print(f'[skip] folder does not exist: {base_folder}')
        planning_skips += 1
        continue

    fire_year = resolve_fire_year(base_folder)
    fire_date = resolve_fire_date(base_folder)

    for zip_path in gather_zip_candidates_for_classification(base_folder):
        try:
            acq_date = parse_acquisition_date_from_name(zip_path)
        except Exception as exc:
            print(f'[skip] could not parse date from {zip_path.name}: {exc}')
            planning_skips += 1
            continue

        bucket = classify_prefire_postfire(acq_date, fire_year, fire_date)
        dest_dir = base_folder / bucket
        dest_zip = dest_dir / zip_path.name

        if zip_path == dest_zip:
            already_in_correct_bucket += 1
            continue

        move_plan.append((zip_path, dest_zip, acq_date.date(), fire_year, fire_date, bucket))
        total_move_zip_bytes += zip_path.stat().st_size

# Report plan details before moving anything.
prefire_count = sum(1 for *_, bucket in move_plan if bucket == 'prefire')
postfire_count = sum(1 for *_, bucket in move_plan if bucket == 'postfire')
print(f'Planned ZIP moves: {len(move_plan)}')
print(f'  prefire={prefire_count}, postfire={postfire_count}, skipped={planning_skips}')
print(f'  already_in_correct_bucket={already_in_correct_bucket}')
print(f'  compressed size to move: {human_size(total_move_zip_bytes)}')

for src, dst, acq_date, fire_year, fire_date, bucket in move_plan[:20]:
    fire_marker = fire_date.isoformat() if fire_date else f'year={fire_year}'
    print(f'  {src} -> {dst} | acq_date={acq_date}, fire={fire_marker}, bucket={bucket}')
if len(move_plan) > 20:
    print(f'  ... and {len(move_plan)-20} more')

# Safety checks before major file operations.
if not DRY_RUN and len(move_plan) > MAX_MOVE_FILES:
    raise RuntimeError(
        f'Planned move count ({len(move_plan)}) exceeds MAX_MOVE_FILES ({MAX_MOVE_FILES}).'
    )

if not DRY_RUN:
    moved = 0
    destination_exists = 0
    for src, dst, *_ in move_plan:
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists():
            destination_exists += 1
            print(f'[skip] destination exists: {dst}')
            continue
        shutil.move(str(src), str(dst))
        moved += 1
        print(f'[move] {src} -> {dst}')
    print(f'Move step complete. moved={moved}, destination_exists={destination_exists}')
else:
    print('DRY_RUN=True, no files moved yet.')

Planned ZIP moves: 188
  prefire=94, postfire=94, skipped=0
  already_in_correct_bucket=0
  compressed size to move: 77.90 GB
  downloads/creek/naip/m_3611906_ne_11_060_20200803.ZIP -> downloads/creek/naip/prefire/m_3611906_ne_11_060_20200803.ZIP | acq_date=2020-08-03, fire=year=2020, bucket=prefire
  downloads/creek/naip/m_3611906_ne_11_060_20220624.ZIP -> downloads/creek/naip/postfire/m_3611906_ne_11_060_20220624.ZIP | acq_date=2022-06-24, fire=year=2020, bucket=postfire
  downloads/creek/naip/m_3611906_nw_11_060_20200803.ZIP -> downloads/creek/naip/prefire/m_3611906_nw_11_060_20200803.ZIP | acq_date=2020-08-03, fire=year=2020, bucket=prefire
  downloads/creek/naip/m_3611906_nw_11_060_20220624.ZIP -> downloads/creek/naip/postfire/m_3611906_nw_11_060_20220624.ZIP | acq_date=2022-06-24, fire=year=2020, bucket=postfire
  downloads/creek/naip/m_3711833_sw_11_060_20200804.ZIP -> downloads/creek/naip/prefire/m_3711833_sw_11_060_20200804.ZIP | acq_date=2020-08-04, fire=year=2020, bucket=pre

In [9]:
def unzip_into_named_folder(zip_path: Path, overwrite: bool = False) -> Path:
    extract_dir = zip_path.with_suffix('')
    if extract_dir.exists() and any(extract_dir.iterdir()) and not overwrite:
        return extract_dir

    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_dir)
    return extract_dir


zip_targets = []
bad_archives = []
total_compressed = 0
total_uncompressed_est = 0

# Preflight unzip checks and storage impact estimates.
for base_folder in NAIP_FOLDERS:
    base_folder = Path(base_folder)
    for bucket in ['prefire', 'postfire']:
        for zip_path in find_bucket_zip_files(base_folder, bucket):
            try:
                compressed, uncompressed = zip_storage_stats(zip_path)
            except zipfile.BadZipFile:
                bad_archives.append(zip_path)
                continue
            except Exception as exc:
                print(f'[warn] unable to inspect {zip_path}: {exc}')
                bad_archives.append(zip_path)
                continue

            zip_targets.append(zip_path)
            total_compressed += compressed
            total_uncompressed_est += uncompressed

print(f'ZIPs available for unzip: {len(zip_targets)}')
print(f'  compressed size: {human_size(total_compressed)}')
print(f'  estimated uncompressed size: {human_size(total_uncompressed_est)}')
if total_compressed > 0:
    ratio = total_uncompressed_est / total_compressed
    print(f'  estimated expansion ratio: {ratio:.2f}x')

if bad_archives:
    print(f'Bad/Unreadable ZIP files: {len(bad_archives)}')
    for path in bad_archives[:20]:
        print(f'  [bad-zip] {path}')
    if len(bad_archives) > 20:
        print(f'  ... and {len(bad_archives)-20} more')

if not DRY_RUN and bad_archives:
    raise RuntimeError('Found bad ZIP archives. Fix/remove them before unzipping.')

if not DRY_RUN and len(zip_targets) > MAX_UNZIP_FILES:
    raise RuntimeError(
        f'Planned unzip count ({len(zip_targets)}) exceeds MAX_UNZIP_FILES ({MAX_UNZIP_FILES}).'
    )

if not DRY_RUN and (total_uncompressed_est / (1024 ** 3)) > MAX_ESTIMATED_UNZIP_GB:
    raise RuntimeError(
        'Estimated uncompressed size exceeds MAX_ESTIMATED_UNZIP_GB. '
        f'Estimate={(total_uncompressed_est / (1024 ** 3)):.2f} GB, limit={MAX_ESTIMATED_UNZIP_GB:.2f} GB.'
    )

unzipped = 0
already_present = 0

for zip_path in zip_targets:
    extract_dir = zip_path.with_suffix('')
    if extract_dir.exists() and any(extract_dir.iterdir()):
        already_present += 1
        print(f'[skip] already extracted: {extract_dir}')
        continue

    if DRY_RUN:
        print(f'[dry-run unzip] {zip_path} -> {extract_dir}')
        continue

    out_dir = unzip_into_named_folder(zip_path, overwrite=False)
    unzipped += 1
    print(f'[unzipped] {zip_path} -> {out_dir}')

if DRY_RUN:
    print('DRY_RUN=True, no ZIPs were extracted.')
else:
    print(f'Completed unzip step. unzipped={unzipped}, already_present={already_present}')

ZIPs available for unzip: 188
  compressed size: 77.90 GB
  estimated uncompressed size: 88.09 GB
  estimated expansion ratio: 1.13x
[unzipped] downloads/creek/naip/prefire/m_3711946_nw_11_060_20200803.ZIP -> downloads/creek/naip/prefire/m_3711946_nw_11_060_20200803
[unzipped] downloads/creek/naip/prefire/m_3711961_se_11_060_20200803.ZIP -> downloads/creek/naip/prefire/m_3711961_se_11_060_20200803
[unzipped] downloads/creek/naip/prefire/m_3711948_nw_11_060_20200804.ZIP -> downloads/creek/naip/prefire/m_3711948_nw_11_060_20200804
[unzipped] downloads/creek/naip/prefire/m_3711922_se_11_060_20200803.ZIP -> downloads/creek/naip/prefire/m_3711922_se_11_060_20200803
[unzipped] downloads/creek/naip/prefire/m_3711946_ne_11_060_20200803.ZIP -> downloads/creek/naip/prefire/m_3711946_ne_11_060_20200803
[unzipped] downloads/creek/naip/prefire/m_3711948_ne_11_060_20200804.ZIP -> downloads/creek/naip/prefire/m_3711948_ne_11_060_20200804
[unzipped] downloads/creek/naip/prefire/m_3711953_sw_11_060_202

## Usage

1. Update `NAIP_FOLDERS` and `FIRE_YEAR_BY_FOLDER` in Cell 3.
2. Optional: set exact fire dates in `FIRE_DATE_BY_FOLDER` (YYYY-MM-DD) for best accuracy.
3. If not using exact dates, set `SAME_YEAR_IS_PREFIRE` in Cell 3.
4. Adjust safety thresholds in Cell 3 (`MAX_MOVE_FILES`, `MAX_UNZIP_FILES`, `MAX_ESTIMATED_UNZIP_GB`).
5. Run all cells once with `DRY_RUN = True` to preview plans and storage impact.
6. Set `DRY_RUN = False` and run the move/unzip cells again to apply changes.

The move and unzip cells perform preflight sanity checks and print counts plus estimated storage expansion before major operations.